# Comparative CSV: Human vs LLM-generated texts

This notebook produces comparative CSV files that combine **human-written** and **LLM-generated** texts:

1. **Abstracts (formal)** — from available abstracts source CSV (e.g. `sv_abstracts_openai.csv` / `sv_abstracts_generated.csv`)
2. **Comments (informal)** — human Reddit comments + generated comments (same logic as `inspect_reddit_comments.py`)

In [1]:
import os
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    # Walk upwards; prefer the repo root by locating `main.tex`.
    for p in [start] + list(start.parents):
        if (p / "main.tex").exists() and (p / "src").exists():
            return p
    # Fallback: first parent that has a `src` directory.
    for p in [start] + list(start.parents):
        if (p / "src").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

# Support both layouts:
# - new: src/1_data_collection/{human_formal,human_informal,llm_formal,...}
# - legacy: src/data_collection/{formal,informal}
NEW_DATA_DIR = SRC_DIR / "1_data_collection"
if NEW_DATA_DIR.exists():
    DATA_DIR = NEW_DATA_DIR
    FORMAL_DIR = DATA_DIR / "human_formal"
    INFORMAL_DIR = DATA_DIR / "human_informal"
    LLM_FORMAL_DIR = DATA_DIR / "llm_formal" / "abstracts"
    LLM_INFORMAL_DIR = DATA_DIR / "llm_informal"
else:
    DATA_DIR = SRC_DIR / "data_collection"
    FORMAL_DIR = DATA_DIR / "formal"
    INFORMAL_DIR = DATA_DIR / "informal"
    LLM_FORMAL_DIR = DATA_DIR / "llm_formal" / "abstracts"
    LLM_INFORMAL_DIR = DATA_DIR / "llm_informal"


def first_existing_or_search(candidates, fallback_names, label):
    checked = []
    for candidate in candidates:
        checked.append(candidate)
        if candidate.exists():
            return candidate

    for name in fallback_names:
        for match in SRC_DIR.rglob(name):
            if match.is_file():
                return match

    raise FileNotFoundError(
        f"Could not find {label}. Checked: {[str(p) for p in checked]} and recursive search for {fallback_names} under {SRC_DIR}"
    )


def maybe_find_or_search(candidates, fallback_names):
    for candidate in candidates:
        if candidate.exists():
            return candidate

    for name in fallback_names:
        for match in SRC_DIR.rglob(name):
            if match.is_file():
                return match

    return None


# Input files
ABSTRACTS_CSV = first_existing_or_search(
    [
        FORMAL_DIR / "sv_abstracts_openai.csv",
        FORMAL_DIR / "sv_abstracts_generated.csv",
        LLM_FORMAL_DIR / "sv_abstracts_openai_2.csv",
    ],
    ["sv_abstracts_openai.csv", "sv_abstracts_generated.csv", "sv_abstracts_openai_2.csv"],
    "abstracts CSV",
)

# Comments inputs are optional; the notebook should skip that block if missing.
COMMENTS_HUMAN_CSV = maybe_find_or_search(
    [
        INFORMAL_DIR / "reddit_comments.csv",
        # legacy layout
        DATA_DIR / "informal" / "reddit_comments.csv",
        SRC_DIR / "2_text_analysis" / "informal" / "reddit_comments.csv",
    ],
    ["reddit_comments.csv"],
)

COMMENTS_GENERATED_CSV = maybe_find_or_search(
    [
        LLM_INFORMAL_DIR / "reddit_comments_openai_100.csv",
        LLM_INFORMAL_DIR / "reddit_comments_openai.csv",
        # legacy layout
        DATA_DIR / "reddit_comments_openai_100.csv",
    ],
    ["reddit_comments_openai_100.csv", "reddit_comments_openai.csv"],
)

# Output files (comparative CSVs)
OUT_ABSTRACTS_COMPARATIVE = FORMAL_DIR / "comparative_abstracts.csv"
OUT_COMMENTS_COMPARATIVE = DATA_DIR / "comparative_comments.csv"

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Abstracts input:", ABSTRACTS_CSV)
print("Comments human:", COMMENTS_HUMAN_CSV)
print("Comments generated:", COMMENTS_GENERATED_CSV)
print("Abstracts output:", OUT_ABSTRACTS_COMPARATIVE)
print("Comments output:", OUT_COMMENTS_COMPARATIVE)

Data dir: c:\Users\GinaWelsh\OneDrive - Tullius\Thesis\CLUU-thesis\src\1_data_collection\src\data_collection
Abstracts input: c:\Users\GinaWelsh\OneDrive - Tullius\Thesis\CLUU-thesis\src\1_data_collection\src\data_collection\formal\sv_abstracts_generated.csv
Comments human: c:\Users\GinaWelsh\OneDrive - Tullius\Thesis\CLUU-thesis\src\1_data_collection\src\data_collection\informal\reddit_comments.csv
Comments generated: c:\Users\GinaWelsh\OneDrive - Tullius\Thesis\CLUU-thesis\src\1_data_collection\src\data_collection\reddit_comments_openai_100.csv


## 1. Abstracts (formal): human vs generated

Load the available abstracts CSV (supports current and legacy filenames, e.g. `sv_abstracts_openai.csv` / `sv_abstracts_generated.csv`). We produce:
- **Wide format** (unchanged): one row per thesis with `Abstract` and `Generated_Abstract`.
- **Long format** (optional): one row per text with `source` = human | generated, for consistent analysis with comments.

In [2]:
# Load formal abstracts (human + generated in same file)
df_abs = pd.read_csv(ABSTRACTS_CSV, encoding="utf-8")
print(f"Loaded {len(df_abs)} rows. Columns: {list(df_abs.columns)}")

# Keep key columns for comparative CSV (human vs LLM side by side)
cols_abstracts = [c for c in ["Year", "Level", "Title", "Topic Category", "Abstract", "Generated_Abstract"] if c in df_abs.columns]
df_abstracts_comparative = df_abs[cols_abstracts].copy()
df_abstracts_comparative.to_csv(OUT_ABSTRACTS_COMPARATIVE, index=False, encoding="utf-8")
print(f"Saved comparative abstracts (wide) to {OUT_ABSTRACTS_COMPARATIVE}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\GinaWelsh\\OneDrive - Tullius\\Thesis\\CLUU-thesis\\src\\1_data_collection\\src\\data_collection\\formal\\sv_abstracts_generated.csv'

In [ ]:
# Optional: long-format abstracts (one row per text, with source = human | generated)
rows_long = []
for idx, row in df_abs.iterrows():
    if pd.notna(row.get("Abstract")) and str(row["Abstract"]).strip():
        rows_long.append({
            "index": idx,
            "Year": row.get("Year"),
            "Title": row.get("Title"),
            "text": row["Abstract"],
            "source": "human",
        })
    if pd.notna(row.get("Generated_Abstract")) and str(row["Generated_Abstract"]).strip():
        rows_long.append({
            "index": idx,
            "Year": row.get("Year"),
            "Title": row.get("Title"),
            "text": row["Generated_Abstract"],
            "source": "generated",
        })
df_abstracts_long = pd.DataFrame(rows_long)
out_abstracts_long = os.path.join(FORMAL_DIR, "comparative_abstracts_long.csv")
df_abstracts_long.to_csv(out_abstracts_long, index=False, encoding="utf-8")
print(f"Saved long-format abstracts to {out_abstracts_long} ({len(df_abstracts_long)} rows)")
df_abstracts_long.head(6)

## 2. Comments (informal): human vs generated

Same logic as `inspect_reddit_comments.get_combined_df()`: load human and generated comment CSVs, add a `source` column, concatenate, and save one comparative CSV.

In [ ]:
# Load human and generated comments (optional block)
if COMMENTS_HUMAN_CSV is None or COMMENTS_GENERATED_CSV is None:
    print("Skipping comments comparative CSV: missing human and/or generated Reddit comments files.")
else:
    human = pd.read_csv(COMMENTS_HUMAN_CSV, encoding="utf-8")
    gen = pd.read_csv(COMMENTS_GENERATED_CSV, encoding="utf-8")
    human["source"] = "human"
    gen["source"] = "generated"

    # Same columns in both (link_id, question, comment, source)
    cols = [c for c in human.columns if c in gen.columns]
    df_comments_comparative = pd.concat([human[cols], gen[cols]], ignore_index=True)
    df_comments_comparative.to_csv(OUT_COMMENTS_COMPARATIVE, index=False, encoding="utf-8")
    print(f"Human comments: {len(human)}, Generated: {len(gen)}")
    print(f"Saved comparative comments to {OUT_COMMENTS_COMPARATIVE} ({len(df_comments_comparative)} rows)")
    df_comments_comparative.head(10)

## Summary

- **Abstracts (wide):** `src/1_data_collection/human_formal/comparative_abstracts.csv` — Year, Level, Title, Topic Category, Abstract, Generated_Abstract  
- **Abstracts (long):** `src/1_data_collection/human_formal/comparative_abstracts_long.csv` — index, Year, Title, text, source  
- **Comments:** `src/1_data_collection/comparative_comments.csv` — link_id, question, comment, source